# Exploratory questions: how Madrid's bus fleet works and how it could transition
**IE Sustainability Datathon (Kearney) · Supporting notebook 03**

Works through the six questions the team agreed on the whiteboard. It uses the same setup cells and `data_raw/` files as the main notebook. Findings feed the main notebook and the report. **This notebook never writes `File_1.csv`–`File_4.csv`.**

Sections 2–5 follow the EDA steps (overview → data preparation → features → hypotheses) and produce the clean tables the questions use. Each question is tagged **[ONCE]** (Madrid only) or **[PER CITY]** (repeated for each benchmark city), as on the board.

## Scope agreed on the whiteboard

| # | Question | Scope | Sub-questions |
|---|---|---|---|
| Q1 | How does the current fleet work? | ONCE | **Buses:** km (range), type (standard / articulated), capacity, gas / electric, age · **Lines:** electric / gas, # buses, riders, km, schedule · **Depots:** grid capacity, electrical output |
| Q2 | Apply the Madrid setup to each benchmark city | PER CITY | KPI differences · time / phases (% electrified = # zero-emission lines ÷ total lines) · gap between Madrid and the city |
| Q3 | Why were certain lines electrified first? | ONCE | Features: depot, similarity of lines, distance, traffic, # buses, riders |
| Q4 | How would we transition the buses? | ONCE | **a)** grid / depots: depot information, gap to electrification per depot, route impact / coverage · **b)** which routes are replaced: phases, route impact (buses, bus metrics) |
| Q5 | What does implementing electrification mean? | ONCE | Finance / costs: bus age and retirement, gas, labour |
| Q6 | Scenario comparison | ONCE | Depot → line · area / zones · bus ages / lifetime · distance · emissions · ridership |

**Data groups:** Madrid's existing setup (km, fleet, lines, depots and grid capacity, emissions) · benchmark cities · external signals (traffic, incline) · financials (age / retirement, gas, labour).
**Constraints:** zero-emission routes · ridership.
**Out of scope** (crossed out on the board): weather, trends / patterns, disposal, backups.

## 0 · Setup
Same helpers and fixed parameters as the main notebook (`fetch`, `assume`, `check`, `P`, GTFS helpers). Keep them identical: if you change one here, change it in notebook 01 too.

In [1]:
# Environment: runs as-is in Google Colab (Python 3.10+).
%pip install -q pandas numpy pyproj shapely folium plotly kaleido requests

import os, io, json, zipfile, datetime as dt, hashlib, textwrap
import numpy as np, pandas as pd, requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

BASE = "/content/datathon" if os.path.exists("/content") else os.path.abspath("..")
RAW, DERIVED, OUT = (os.path.join(BASE, d) for d in ("data_raw", "data_derived", "outputs"))
for d in (RAW, DERIVED, OUT):
    os.makedirs(d, exist_ok=True)

RUN_DATE = dt.date.today().isoformat()
import sys, platform
print(f"Run date {RUN_DATE} | Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}")
print("Folders: data_raw/, data_derived/, outputs/ under", "/content/datathon (Colab)" if BASE.startswith("/content") else "the repo root")

Note: you may need to restart the kernel to use updated packages.


Run date 2026-11-26 | Python 3.13.1 | pandas 2.2.3 | numpy 2.4.6
Folders: data_raw/, data_derived/, outputs/ under the repo root


**Helpers.** `fetch()` downloads each source once and logs URL, date, size and checksum; `assume()` labels every value V, D, P or A for the assumptions register; `check()` prints a visible PASS / FAIL line.

In [2]:
# ---- Data register: every downloaded or manually obtained source is logged here.
DATA_REGISTER = []

def fetch(url, filename, note="", manual=False):
    """Download a public file into data_raw/ and log source, date, size and checksum.
    manual=True: the file was obtained by hand (registration/API); it must already be in data_raw/."""
    path = os.path.join(RAW, filename)
    if not manual and not os.path.exists(path):
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    size = os.path.getsize(path) if os.path.exists(path) else None
    sha = hashlib.sha256(open(path, "rb").read()).hexdigest()[:12] if size else None
    DATA_REGISTER.append(dict(file=filename, url=url, downloaded=RUN_DATE, bytes=size,
                              sha256_12=sha, obtained="manual" if manual else "script", note=note))
    return path

def show_register():
    display(pd.DataFrame(DATA_REGISTER))

# ---- Provenance labels (brief section 4.6): V verified, D derived, P proxy, A assumption.
ASSUMPTIONS = []

def assume(variable, value, label, evidence, impact="", sensitivity=""):
    assert label in {"V", "D", "P", "A"}, "label must be V, D, P or A"
    ASSUMPTIONS.append(dict(variable=variable, value=value, label=label, evidence=evidence,
                            impact=impact, sensitivity=sensitivity))
    return value

# ---- Visible checks: every step prints CHECK ... PASS / FAIL.
CHECKS = []

def check(name, ok, detail=""):
    CHECKS.append(dict(check=name, result="PASS" if ok else "FAIL", detail=detail))
    print(f"CHECK {'PASS' if ok else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    return ok
print("Helpers ready: fetch() logs sources, assume() labels values V/D/P/A, check() prints PASS/FAIL")


Helpers ready: fetch() logs sources, assume() labels values V/D/P/A, check() prints PASS/FAIL


**Fixed parameters (brief §4.5).** Read from notebook 01's parameter cell, so the notebooks can never disagree on a fixed value.

In [3]:
# The brief's fixed parameters are read out of notebook 01's own parameter cell, never copied here.
def load_P_from_main_notebook():
    path = os.path.join(BASE, "notebooks", "01_Madrid_Electrification_Main.ipynb")
    for c in json.load(open(path))["cells"]:
        s = "".join(c["source"])
        if c["cell_type"] == "code" and "P = dict(" in s:
            g = {"assume": assume, "pd": pd, "np": np, "display": lambda *a, **k: None}
            exec(s, g)
            return g["P"]
    raise RuntimeError("parameter cell 'P = dict(...)' not found in notebook 01")

P = load_P_from_main_notebook()
display(pd.DataFrame([(k, v) for k, v in P.items()], columns=["parameter", "value"]))


,parameter,value
0,kwh_km,"{'Standard': 1.3, 'Articulated': 1.85}"
1,range_km,"{'Standard': 300, 'Articulated': 240}"
2,cng_kg_km,"{'Standard': 0.45, 'Articulated': 0.65}"
3,charger_kw,100
4,window_h,7
5,charge_eff,0.9
6,bus_price_eur,"{'Standard': 615000, 'Articulated': 800000}"
7,benchmark_kwh_km,1.3
8,service_days,365
9,cng_kgco2_per_kg,2.7175


**GTFS helpers.** Read a feed, find the services active on a date, and measure trip length from `shape_dist_traveled` or the shape geometry.

In [4]:
# ---- Generic GTFS helpers (shared logic with the benchmark notebook).
def read_gtfs(zip_path, tables=("agency","routes","trips","stop_times","calendar","calendar_dates","shapes","feed_info")):
    feed = {}
    with zipfile.ZipFile(zip_path) as z:
        names = set(z.namelist())
        for t in tables:
            if f"{t}.txt" in names:
                feed[t] = pd.read_csv(z.open(f"{t}.txt"), dtype=str, low_memory=False)
    return feed

def active_service_ids(feed, date_str):
    """Service ids running on date_str (YYYY-MM-DD), applying calendar then calendar_dates exceptions."""
    d = pd.Timestamp(date_str); ymd = d.strftime("%Y%m%d"); dow = d.day_name().lower()
    active = set()
    if "calendar" in feed:
        c = feed["calendar"]
        m = (c["start_date"] <= ymd) & (c["end_date"] >= ymd) & (c[dow] == "1")
        active |= set(c.loc[m, "service_id"])
    if "calendar_dates" in feed:
        cd = feed["calendar_dates"]; cd = cd[cd["date"] == ymd]
        active |= set(cd.loc[cd["exception_type"] == "1", "service_id"])
        active -= set(cd.loc[cd["exception_type"] == "2", "service_id"])
    return active

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(a))

def shape_lengths_km(feed):
    s = feed["shapes"].copy()
    for c in ("shape_pt_lat","shape_pt_lon"): s[c] = s[c].astype(float)
    s["shape_pt_sequence"] = s["shape_pt_sequence"].astype(int)
    s = s.sort_values(["shape_id","shape_pt_sequence"])
    g = s.groupby("shape_id")
    s["seg"] = haversine_km(g["shape_pt_lat"].shift(), g["shape_pt_lon"].shift(), s["shape_pt_lat"], s["shape_pt_lon"])
    return s.groupby("shape_id")["seg"].sum()

def trip_km(feed, trip_ids, dist_unit_m=None):
    """Km per trip. Preferred: shape_dist_traveled (unit must be confirmed: metres or km);
    fallback: geodesic length of the trip's shape. Prints which method each trip used."""
    st = feed["stop_times"][feed["stop_times"]["trip_id"].isin(trip_ids)]
    km = pd.Series(dtype=float)
    if "shape_dist_traveled" in st and st["shape_dist_traveled"].notna().any():
        d = st.assign(x=pd.to_numeric(st["shape_dist_traveled"], errors="coerce")).groupby("trip_id")["x"].agg(lambda v: v.max() - v.min())
        assert dist_unit_m is not None, "Confirm shape_dist_traveled unit (metres or km) and pass dist_unit_m=True/False"
        km = d / 1000 if dist_unit_m else d
    missing = set(trip_ids) - set(km.dropna().index)
    if missing and "shapes" in feed:
        lens = shape_lengths_km(feed)
        t = feed["trips"].set_index("trip_id").loc[list(missing), "shape_id"]
        km = pd.concat([km.dropna(), t.map(lens)])
    print(f"trip_km: {len(km)} trips, {len(missing)} via shape geometry fallback")
    return km
print("GTFS helpers ready: read_gtfs, active_service_ids, trip-length functions")


GTFS helpers ready: read_gtfs, active_service_ids, trip-length functions


## 1 · Data
Same sources as main notebook §1. `fetch()` reuses files already in `data_raw/`.

In [5]:
# Direct file URLs, resolved from each landing page via the portal's CKAN API (brief section 4.1).
EMT = "https://datos.emtmadrid.es/dataset"
SOURCES = {
    # filename: (url, note)
    "emt_gtfs.zip": (f"{EMT}/9b23259a-4491-494b-9695-36a7709b2c12/resource/3cba2058-9833-422c-a704-bf992d31d2ee/download/gtfs_emt.zip",
                     "EMT GTFS, rolling daily feed — https://datos.madrid.es/dataset/900030-0-emt-autobus-gtfs"),
    "emt_supply_2026.csv": (f"{EMT}/03c64a21-3ee8-461a-9f9a-cb4c727a41be/resource/8400b44b-ca21-45b9-b3cf-7a15b754b739/download/2026busoferta.csv",
                     "Buses in service per day, line and hour band, 2026 — https://datos.madrid.es/dataset/300596-0-emt-lineas-autobuses-tramos"),
    "emt_fleet_history.csv": (f"{EMT}/071db69e-5e59-4ceb-a6da-92d895f56b62/resource/48d59bf3-93a8-48b7-96bd-cf78cc7c902b/download/vehicles.csv",
                     "Fleet history (vehicle, dates, fuel, size) — https://datos.madrid.es/dataset/300241-0-emt-flota-autobuses"),
    "emt_calendar.csv": (f"{EMT}/405a6ed0-9464-4d04-9edb-939031717b20/resource/b8cceab8-9173-4eec-8431-21f5e9e25834/download/calendar.csv",
                     "Operating calendar 2024-2027 — https://datos.madrid.es/dataset/900022-0-calendario-emt-autobus"),
    "emt_ridership_2026.csv": (f"{EMT}/29b27217-429f-44a0-a90a-2db7f13529b6/resource/07622d4b-6c64-43ea-a398-69134d6128bc/download/2026demdialineafranjaanual.csv",
                     "Optional: daily riders per line and hour band, 2026 — https://datos.madrid.es/dataset/900028-0-emt-demanda-autobus"),
}
PATHS = {fn: fetch(url, fn, note) for fn, (url, note) in SOURCES.items()}
show_register()

,file,url,downloaded,bytes,sha256_12,obtained,note
0,emt_gtfs.zip,https://datos.emtmadrid.es/dataset/9b23259a-44...,2026-11-26,21044007,62453fb95492,script,"EMT GTFS, rolling daily feed — https://datos.m..."
1,emt_supply_2026.csv,https://datos.emtmadrid.es/dataset/03c64a21-3e...,2026-11-26,21555684,9fdcee98ef74,script,"Buses in service per day, line and hour band, ..."
2,emt_fleet_history.csv,https://datos.emtmadrid.es/dataset/071db69e-5e...,2026-11-26,1109695,ff90c42290bd,script,"Fleet history (vehicle, dates, fuel, size) — h..."
3,emt_calendar.csv,https://datos.emtmadrid.es/dataset/405a6ed0-94...,2026-11-26,56877,b967eda5e8d5,script,Operating calendar 2024-2027 — https://datos.m...
4,emt_ridership_2026.csv,https://datos.emtmadrid.es/dataset/29b27217-42...,2026-11-26,24395594,d3bffd83ce3e,script,"Optional: daily riders per line and hour band,..."


## 2 · EDA step 1: connect the data and get an overview
Read every source exactly as published (all columns as text, nothing changed yet): size, duplicates, missing cells, then a column-by-column profile. This is what we are dealing with before any cleaning.

In [6]:
# Raw read: strings only. The calendar is read with keep_default_na=False because 'NA' is a real day-type code (step 2a).
import plotly.express as px
raw = {
    "fleet":    pd.read_csv(PATHS["emt_fleet_history.csv"], dtype=str),
    "supply":   pd.read_csv(PATHS["emt_supply_2026.csv"], sep=";", dtype=str),
    "riders":   pd.read_csv(PATHS["emt_ridership_2026.csv"], sep=";", dtype=str),
    "calendar": pd.read_csv(PATHS["emt_calendar.csv"], dtype=str, keep_default_na=False),
}

def overview(df):
    return dict(rows=len(df), columns=df.shape[1], duplicate_rows=int(df.duplicated().sum()),
                missing_cells=int(df.isna().sum().sum()), memory_mb=round(df.memory_usage(deep=True).sum() / 1e6, 1))

display(pd.DataFrame({name: overview(df) for name, df in raw.items()}).T)
with zipfile.ZipFile(PATHS["emt_gtfs.zip"]) as z:
    gtfs_rows = {n[:-4]: sum(1 for _ in z.open(n)) - 1 for n in z.namelist()}
display(pd.Series(gtfs_rows, name="rows").to_frame().T)

,rows,columns,duplicate_rows,missing_cells,memory_mb
fleet,12920.0,11.0,5675.0,0.0,7.9
supply,939717.0,5.0,0.0,0.0,247.1
riders,1026041.0,4.0,0.0,0.0,220.4
calendar,1461.0,10.0,0.0,0.0,0.8


,routes,stops,trips,frequencies,calendar_dates,calendar,agency,stop_times,shapes
rows,238,4934,86635,86599,8,5,1,2226330,194888


**Column profile.** For each dataset: the first rows, then each column's type, missing values, distinct values and a few examples.

In [7]:
def profile(df):
    return pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(), "unique": df.nunique(),
                         "examples": [", ".join(map(str, df[c].dropna().unique()[:6])) for c in df.columns]})

for name, df in raw.items():
    print(f"\n=== {name}: first rows and column profile")
    display(df.head(3)); display(profile(df))


=== fleet: first rows and column profile


,vehicleId,dateIni,dateEnd,trade,model,version,typeCapacity,fuelType,accesibility,capacity,colour
0,0051,2023-04-11 00:00:00,9999-12-31 00:00:00,IRIZAR,IETRAM,0,01,EL,04,80,B
1,0052,2023-03-25 00:00:00,9999-12-31 00:00:00,IRIZAR,IETRAM,0,01,EL,04,80,B
2,0053,2023-05-05 00:00:00,9999-12-31 00:00:00,IRIZAR,IETRAM,0,01,EL,04,80,B


,dtype,missing,unique,examples
vehicleId,object,0,5351,"0051, 0052, 0053, 0054, 0055, 0056"
dateIni,object,0,1423,"2023-04-11 00:00:00, 2023-03-25 00:00:00, 2023..."
dateEnd,object,0,925,"9999-12-31 00:00:00, 2004-07-16 00:00:00, 2017..."
trade,object,0,24,"IRIZAR , IRIZAR, PEGASO , IVECO , MA..."
model,object,0,68,"IETRAM , IETRAM, 6424 , I2E , IE..."
version,object,0,12,"0, 7, 1, 2, 3, 4"
typeCapacity,object,0,6,"01 , 02 , 05 , 03 , 06 , 04"
fuelType,object,0,6,"EL , GB , GN , ET , DE , H"
accesibility,object,0,5,"04 , 01 , 03 , 4 , 02"
capacity,object,0,35,"80, 51, 77, 87, 120, 118"



=== supply: first rows and column profile


,Linea,FechaSer,TipoDiaMo,CodFranja,NumCCReal
0,001,2026-01-01,FE,7,2
1,001,2026-01-01,FE,8,3
2,001,2026-01-01,FE,9,3


,dtype,missing,unique,examples
Linea,object,0,246,"001, 002, 003, 004, 005, 006"
FechaSer,object,0,243,"2026-01-01, 2026-01-02, 2026-01-03, 2026-01-04..."
TipoDiaMo,object,0,3,"FE, LA, SA"
CodFranja,object,0,32,"7, 8, 9, 10, 11, 12"
NumCCReal,object,0,30,"2, 3, 4, 8, 10, 9"


... (2 outputs left out)

,dtype,missing,unique,examples
Fecha,object,0,273,"2026-01-01, 2026-01-02, 2026-01-03, 2026-01-04..."
Linea,object,0,254,"001, 002, 003, 004, 005, 006"
Franja,object,0,32,"H07, H08, H09, H10, H11, H12"
TotalViajeros,object,0,3243,"17, 21, 26, 76, 95, 131"



=== calendar: first rows and column profile


,Fecha,TipoDiaEs,TemporadaTG,DiaSemana,Semana,Mes,Trimestre,Clima,TemperaturaMax,TemperaturaMin
0,2027-12-31,NA,IV,V,53,12,4,BBB,0.0,0.0
1,2027-12-30,NA,IV,J,53,12,4,BBB,0.0,0.0
2,2027-12-29,NA,IV,X,53,12,4,BBB,0.0,0.0


,dtype,missing,unique,examples
Fecha,object,0,1461,"2027-12-31, 2027-12-30, 2027-12-29, 2027-12-28..."
TipoDiaEs,object,0,8,"NA, LA, FE, SA, SS, HU"
TemporadaTG,object,0,4,"IV, VN, IM, BJ"
DiaSemana,object,0,7,"V, J, X, M, L, D"
Semana,object,0,53,"53, 52, 51, 50, 49, 48"
Mes,object,0,12,"12, 11, 10, 9, 8, 7"
Trimestre,object,0,4,"4, 3, 2, 1"
Clima,object,0,5,"BBB, , Pr , SPR, LLU"
TemperaturaMax,object,0,292,"0.0, 1.2, 25.9, 27.8, 28.0, 27.7"
TemperaturaMin,object,0,245,"0.0, 0.1, 14.8, 13.4, 15.1, 13.7"


## 3 · EDA step 2: data preparation
What step 1 showed, and what we do about it:

| Issue | Where | Action |
|---|---|---|
| Text padded with spaces (`'EL '`, `'IRIZAR    '`) | fleet | strip all text |
| 44 % of rows are exact duplicates; the rest still has several versions of the same vehicle (same dates, different capacity, colour or accessibility) | fleet | drop exact duplicates; keep one record per vehicle |
| `'NA'` is a day-type code (Christmas period), not a missing value | calendar | read with `keep_default_na=False` and label it |
| `'?'` colour, blank weather, `0.0` temperatures | fleet, calendar | columns dropped (not used; weather is out of scope) |
| Hour band is `'H07'` / `'M01'` (after midnight, night lines) in ridership, `7` / `25` in supply | riders | convert to the supply scale: `H`*nn* → *nn*, `M`*nn* → 24 + *nn* |
| Spanish column names and codes | all | rename to English `snake_case`, add readable labels |
| Lines with riders but no supply records | riders vs supply | keep, but flag: they cannot get a peak-bus count |

### 3a · Reading issues

In [8]:
na_days = sorted(raw["calendar"].loc[raw["calendar"].TipoDiaEs == "NA", "Fecha"])
print(f"Calendar: {len(na_days)} days coded 'NA'; a default read turns them into missing values. Dates: {', '.join(na_days[:9])} ...")
padded = {c: int((raw["fleet"][c] != raw["fleet"][c].str.strip()).sum()) for c in raw["fleet"].columns}
print("Fleet values padded with spaces, per column:", {k: v for k, v in padded.items() if v})

Calendar: 62 days coded 'NA'; a default read turns them into missing values. Dates: 2024-01-01, 2024-01-02, 2024-01-03, 2024-01-04, 2024-01-05, 2024-01-06, 2024-01-07, 2024-12-23, 2024-12-24 ...
Fleet values padded with spaces, per column: {'trade': 12701, 'model': 10315, 'typeCapacity': 12920, 'fuelType': 12920, 'accesibility': 12920}


### 3b · Duplicate rows and records

In [9]:
fleet = raw["fleet"].apply(lambda c: c.str.strip())
n0 = len(fleet); fleet = fleet.drop_duplicates(); n1 = len(fleet)
print(f"Fleet: {n0 - n1} exact duplicate rows removed ({(n0 - n1) / n0:.0%}); {n1} records left")

key = ["vehicleId", "dateIni"]
conflicts = pd.Series({c: int(fleet.groupby(key)[c].nunique().gt(1).sum()) for c in fleet.columns if c not in key},
                      name="vehicle + start-date groups with more than one value")
display(conflicts.to_frame())

fleet["_cap"] = pd.to_numeric(fleet.capacity, errors="coerce")
fleet["_ze"] = fleet.fuelType.isin(["EL", "H"])
fleet = fleet.sort_values(["_cap", "_ze"], ascending=[False, True]).drop_duplicates(key).drop(columns=["_cap", "_ze"])
assume("Fleet near-duplicate rule", "one record per vehicle, highest capacity kept; on a fuel conflict the non-zero-emission record", "A",
       "Near-duplicates differ mainly in capacity and colour. Five 2012 Tempus buses (9301-9305) carry both GN and EL records "
       "with the same dates and capacity: counted as CNG, as in main notebook §3")

check("Fleet: one record per vehicle", fleet.vehicleId.is_unique, f"{len(fleet)} records, {fleet.vehicleId.nunique()} vehicles")
for name, k in {"supply": ["Linea", "FechaSer", "CodFranja"], "riders": ["Fecha", "Linea", "Franja"], "calendar": ["Fecha"]}.items():
    check(f"{name}: no duplicate keys", not raw[name].duplicated(k).any())

Fleet: 5675 exact duplicate rows removed (44%); 7245 records left


,vehicle + start-date groups with more than one value
dateEnd,0
trade,0
model,10
version,14
typeCapacity,6
fuelType,10
accesibility,130
capacity,1429
colour,361


CHECK PASS | Fleet: one record per vehicle | 5351 records, 5351 vehicles
CHECK PASS | supply: no duplicate keys
CHECK PASS | riders: no duplicate keys
CHECK PASS | calendar: no duplicate keys


### 3c · Missing values and placeholders
Counts blanks, `NaN` and `'?'` per column. Zero-rider rows are real observations (very early or late hours) and are kept.

In [10]:
def blanks(df):
    s = df.apply(lambda c: (c.isna() | c.str.strip().isin(["", "?"])).sum())
    return s[s > 0].to_dict() or "none"

for name, df in raw.items():
    print(f"{name}: {blanks(df)}")
zero_rows = int((pd.to_numeric(raw["riders"].TotalViajeros) == 0).sum())
print(f"riders: {zero_rows} line-hour rows with 0 riders (kept)")

fleet: {'colour': 1}


supply: none


riders: none
calendar: {'Clima': 571}


riders: 148 line-hour rows with 0 riders (kept)


### 3d · Rename, recode, set types and drop unused columns
Code labels that are not in a published data dictionary are logged as proxies (**P**).

In [11]:
FUEL = {"GN": "CNG", "EL": "Battery-electric", "H": "Hydrogen fuel cell", "DE": "Diesel", "ET": "Ethanol", "GB": "Unconfirmed (GB)"}
DAY = {"LA": "Working day", "SA": "Saturday", "FE": "Sunday / holiday", "NA": "Christmas period",
       "SS": "Holy Week", "HU": "Strike", "EV": "Event", "PU": "Bridge day"}
assume("Fuel code labels", FUEL, "P", "EL, GN and H match EMT's published fleet mix; DE, ET and GB inferred, no data dictionary")
assume("Zero-emission fuel codes", ("EL", "H"), "A", "Battery-electric and hydrogen only; CNG and biomethane never count (README rule)")
assume("Day-type labels", DAY, "P", "LA, SA, FE documented in the supply README; the others inferred from their dates")

fleet = (fleet.rename(columns={"vehicleId": "vehicle_id", "dateIni": "start_date", "dateEnd": "end_date", "trade": "make",
                               "typeCapacity": "size_code", "fuelType": "fuel_code"})
              .drop(columns=["version", "accesibility", "colour"]))
fleet["start_date"] = pd.to_datetime(fleet.start_date.str[:10])
fleet["end_date"] = pd.to_datetime(fleet.end_date.str[:10].replace("9999-12-31", "2262-04-11"))   # open-ended = still in service
fleet["capacity"] = pd.to_numeric(fleet.capacity).astype("Int64")
fleet["fuel"] = fleet.fuel_code.map(FUEL)
fleet["zero_emission"] = fleet.fuel_code.isin(["EL", "H"])

cal = (raw["calendar"].rename(columns={"Fecha": "date", "TipoDiaEs": "day_code", "TemporadaTG": "season_code",
                                       "DiaSemana": "weekday_code", "Semana": "week", "Mes": "month", "Trimestre": "quarter"})
          .drop(columns=["Clima", "TemperaturaMax", "TemperaturaMin"]))     # weather: out of scope (whiteboard)
cal["date"] = pd.to_datetime(cal.date)
cal["day_type"] = cal.day_code.map(DAY)

supply = raw["supply"].rename(columns={"Linea": "line_id", "FechaSer": "date", "TipoDiaMo": "day_code",
                                       "CodFranja": "hour_band", "NumCCReal": "buses_in_service"})
supply = supply.astype({"hour_band": int, "buses_in_service": int})
supply["date"] = pd.to_datetime(supply.date)

riders = raw["riders"].rename(columns={"Fecha": "date", "Linea": "line_id", "Franja": "hour_band", "TotalViajeros": "riders"})
band = riders.hour_band.str[1:].astype(int)                                      # 'H07' -> 7; 'M01' = 01:00 after midnight -> 25
riders["hour_band"] = np.where(riders.hour_band.str.startswith("M"), band + 24, band)
riders["riders"] = riders.riders.astype(int)
riders["date"] = pd.to_datetime(riders.date)
riders = riders.merge(cal[["date", "day_code"]], on="date", how="left")        # day type from EMT's calendar

agree = supply[["date", "day_code"]].drop_duplicates().merge(cal[["date", "day_code"]], on="date", suffixes=("_supply", "_calendar"))
print("Supply vs calendar day codes (rows = supply, columns = calendar):")
display(pd.crosstab(agree.day_code_supply, agree.day_code_calendar))
only_riders = sorted(set(riders.line_id) - set(supply.line_id))
print(f"Lines with riders but no supply records (kept, flagged): {only_riders}")

check("Every fleet fuel code has a label", fleet.fuel.notna().all())
check("Every calendar day has a label", cal.day_type.notna().all())
check("Every ridership date is in the calendar", riders.day_code.notna().all())
check("Hour bands share one scale (0-31)", riders.hour_band.between(0, 31).all() and supply.hour_band.between(0, 31).all())

Supply vs calendar day codes (rows = supply, columns = calendar):


day_code_calendar,EV,FE,LA,NA,SA,SS
day_code_supply,,,,,,
FE,0,37,0,3,0,3
LA,1,0,161,2,0,3
SA,0,0,0,1,31,1


Lines with riders but no supply records (kept, flagged): ['741', '809', '841', '886', '887', '888', '889', '890']
CHECK PASS | Every fleet fuel code has a label
CHECK PASS | Every calendar day has a label
CHECK PASS | Every ridership date is in the calendar
CHECK PASS | Hour bands share one scale (0-31)


np.True_

### 3e · Plot the data: distributions and outliers
Daily values per line (peak buses, riders) by day type, and capacity by fuel. Outliers are counted with the 1.5 × IQR rule and **kept**: they are real high-demand lines, not errors.

In [12]:
line_day = (supply.groupby(["line_id", "date", "day_code"]).buses_in_service.max().rename("peak_buses").reset_index()
            .merge(riders.groupby(["line_id", "date"]).riders.sum().reset_index(), on=["line_id", "date"], how="left"))

px.box(line_day, x="day_code", y="peak_buses", title="Peak buses per line and day, by day type").show()
px.box(line_day, x="day_code", y="riders", title="Riders per line and day, by day type").show()
px.histogram(fleet.drop_duplicates("vehicle_id", keep="last"), x="capacity", color="fuel", barmode="overlay",
             title="Capacity per vehicle (latest record), by fuel").show()

def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75]); return s[(s < q1 - 1.5 * (q3 - q1)) | (s > q3 + 1.5 * (q3 - q1))]
wd = line_day[line_day.day_code == "LA"]
for col in ("peak_buses", "riders"):
    o = iqr_outliers(wd[col].dropna())
    print(f"Working days, {col}: {len(o)} line-days above/below 1.5 x IQR; lines: {sorted(wd.loc[o.index, 'line_id'].unique())[:12]}")

Working days, peak_buses: 725 line-days above/below 1.5 x IQR; lines: ['019', '021', '027', '028', '034', '049', '068', '069', '133']
Working days, riders: 1285 line-days above/below 1.5 x IQR; lines: ['021', '027', '028', '031', '034', '035', '038', '049', '068', '069', '070', '138']


### 3f · Clean tables
The tables every later section uses. Saved to `data_derived/` (git-ignored) so other notebooks can read them without repeating step 2.

In [13]:
CLEAN = dict(fleet=fleet, calendar=cal, supply=supply, riders=riders)
for name, df in CLEAN.items():
    df.to_csv(os.path.join(DERIVED, f"clean_{name}.csv.gz"), index=False)
display(pd.DataFrame({name: overview(df) for name, df in CLEAN.items()}).T)
for name, df in CLEAN.items():
    print(name, list(df.columns))

,rows,columns,duplicate_rows,missing_cells,memory_mb
fleet,5351.0,10.0,0.0,0.0,1.9
calendar,1461.0,8.0,0.0,0.0,0.5
supply,939717.0,5.0,0.0,0.0,119.3
riders,1026041.0,5.0,0.0,0.0,130.3


fleet ['vehicle_id', 'start_date', 'end_date', 'make', 'model', 'size_code', 'fuel_code', 'capacity', 'fuel', 'zero_emission']
calendar ['date', 'day_code', 'season_code', 'weekday_code', 'week', 'month', 'quarter', 'day_type']
supply ['line_id', 'date', 'day_code', 'hour_band', 'buses_in_service']
riders ['date', 'line_id', 'hour_band', 'riders', 'day_code']


## 4 · EDA step 3: features
One row per line on working days. Built from supply and ridership only; the features that need other sources are listed below the heatmap.

| Feature | Meaning |
|---|---|
| `peak_buses` | max buses in service in any hour band (average over working days) |
| `bus_hours` | sum of buses in service over all hour bands: a proxy for vehicle-hours |
| `span_hours` | number of hour bands with service |
| `peak_to_average` | `peak_buses` ÷ average buses per active hour: how peaky the line is |
| `riders_per_day` | daily riders on working days |
| `riders_per_bus_hour` | productivity: riders ÷ `bus_hours` |
| `weekend_riders_ratio` | average Saturday / Sunday riders ÷ working-day riders |

In [14]:
s = supply[supply.day_code == "LA"]
per_day = s.groupby(["line_id", "date"]).agg(peak_buses=("buses_in_service", "max"), bus_hours=("buses_in_service", "sum"),
                                             span_hours=("hour_band", "nunique"))
features = per_day.groupby("line_id").mean()
rd = riders.groupby(["line_id", "date", "day_code"]).riders.sum().reset_index()
features["riders_per_day"] = rd[rd.day_code == "LA"].groupby("line_id").riders.mean()
features["weekend_riders_ratio"] = rd[rd.day_code.isin(["SA", "FE"])].groupby("line_id").riders.mean() / features.riders_per_day
features["riders_per_bus_hour"] = features.riders_per_day / features.bus_hours
features["peak_to_average"] = features.peak_buses / (features.bus_hours / features.span_hours)
features = features.round(2)

display(features.describe().round(1))
features.to_csv(os.path.join(DERIVED, "line_features.csv"))
check("Line features: every supply line has riders", features.riders_per_day.notna().mean() > 0.95,
      f"{features.riders_per_day.notna().sum()} of {len(features)} lines")

,peak_buses,bus_hours,span_hours,riders_per_day,weekend_riders_ratio,riders_per_bus_hour,peak_to_average
count,240.0,240.0,240.0,240.0,226.0,240.0,240.0
mean,8.1,112.7,17.0,6794.0,0.7,52.7,1.3
std,4.7,72.2,4.5,5862.5,0.4,22.6,0.2
min,1.0,4.0,2.7,47.0,0.0,3.4,1.0
25%,4.3,54.4,17.0,2219.1,0.4,37.8,1.2
50%,7.6,106.7,19.0,5948.9,0.5,52.6,1.2
75%,10.9,152.3,20.0,9469.0,0.7,66.9,1.4
max,25.7,359.9,21.2,32876.0,1.6,144.0,1.7


CHECK PASS | Line features: every supply line has riders | 240 of 240 lines


np.True_

**Feature correlation.** Pairs above |r| = 0.8 carry the same information; keep one of each when weighting lines in §7 of the main notebook.

In [15]:
corr = features.corr()
px.imshow(corr.round(2), text_auto=True, zmin=-1, zmax=1, color_continuous_scale="RdBu_r",
          title="Correlation between line features (working days)").show()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
print("Highly correlated pairs (|r| > 0.8), keep one of each for modelling:")
display(pairs[pairs.abs() > 0.8].round(2).to_frame("r"))

Highly correlated pairs (|r| > 0.8), keep one of each for modelling:


r
peak_buses bus_hours             0.93
           riders_per_day        0.86
bus_hours  riders_per_day        0.92
span_hours weekend_riders_ratio -0.84

**Features still to add** (other sources): daily km and average trip distance (GTFS, main notebook §4) · zero-emission status per line (evidence list: 45 electric + line 145 hydrogen) · depot (main notebook §5) · traffic and incline (external, Q3).

## 5 · EDA step 4: hypotheses
What we expect the data to show, and where each is tested.

| # | Hypothesis | Question | Data | How we test it |
|---|---|---|---|---|
| H1 | Lines electrified first are short, high-ridership lines served by depots that were converted early | Q3 | line features + ZE label + depot | compare feature distributions, electrified vs not |
| H2 | At the 2021–25 purchase pace, Madrid reaches 100 % zero-emission only in the late 2040s | Timeline, Q6 | fleet | pace calculation below |
| H3 | For 2030, depot grid connections bind before bus purchases do | Q4a | depot table, File 3 | required vs available MW per depot |
| H4 | Peaky lines (high `peak_to_average`) need more buses per rider, so they add the most charging demand per depot | Q4b, Q1 | line features, GTFS km | rank lines by buses and energy per rider |
| H5 | A large part of the CNG fleet reaches end of life by 2035, so retirements can carry most of the transition | Q5 | fleet age profile | retirements per year vs buses needed per year |
| H6 | Madrid's gap to peer cities comes from pace, not network size | Q2 | benchmark (notebook 02) | compare ZE share and years-to-target per city |

In [16]:
# Helper for the questions below: vehicles in service on a date, built on the clean fleet table.
def active_fleet(date):
    d = pd.Timestamp(date)
    a = (fleet[(fleet.start_date <= d) & (fleet.end_date > d)]
         .sort_values("start_date").drop_duplicates("vehicle_id", keep="last"))
    first = fleet.groupby("vehicle_id").start_date.min().rename("first_in_service")
    a = a.join(first, on="vehicle_id")
    a["age_years"] = (d - a.first_in_service).dt.days / 365.25
    return a

print(f"Clean fleet: {len(fleet)} records, {fleet.vehicle_id.nunique()} vehicles, last entry {fleet.start_date.max().date()}")

Clean fleet: 5351 records, 5351 vehicles, last entry 2025-06-05


## 6 · Timeline anchors
How the whiteboard's years map onto the brief:

| Anchor | Year | Role in the brief |
|---|---|---|
| S1 target | 2030 | Mandatory File 4 scenario: 100 % zero-emission by 31 Dec 2030, held to 2035 |
| Madrid official target | 2033 | File 1 `ze_target_year`, only if evidenced as a zero-emission (not clean-only) commitment |
| File 4 horizon | 2035 | Last year of File 4; the report must also give the earliest credible year for 100 % |
| S0 at the current pace | computed below | Year the fleet is fully zero-emission at the **unadjusted 2021–25 pace** of purchasing buses per year (brief Objective 3); an insight for the report, since File 4 stops at 2035 |

The pace here is **unadjusted** and the fleet history's last entry is mid-2025, so main notebook §3 reconciles the 31 Dec 2025 baseline against EMT's accounts, and §8 argues the adjusted S0 pace; use those figures in the report.

In [17]:
snap = {d: active_fleet(d) for d in ("2020-12-31", "2025-12-31")}
pace_tbl = pd.DataFrame({d: dict(fleet_total=len(a), zero_emission=int(a.zero_emission.sum()), not_zero_emission=int((~a.zero_emission).sum()))
                         for d, a in snap.items()}).T
display(pace_tbl)

ze_per_year = assume("Zero-emission buses added per year, 2021-25 (unadjusted)",
                     (pace_tbl.loc["2025-12-31", "zero_emission"] - pace_tbl.loc["2020-12-31", "zero_emission"]) / 5,
                     "D", "Fleet history, active vehicles on 31 Dec 2020 and 31 Dec 2025", "Timeline", "Main notebook §3 reconciliation")
remaining = pace_tbl.loc["2025-12-31", "not_zero_emission"]
must_do_year = 2025 + int(np.ceil(remaining / ze_per_year))

TIMELINE = pd.DataFrame([
    ("S1 target", assume("S1 target year", 2030, "V", "Brief scenario S1")),
    ("Madrid official target", assume("Madrid official target year", 2033, "A", "City of Madrid target, Aug 2023; to verify it is zero-emission, not clean-only")),
    ("File 4 horizon", assume("File 4 horizon", 2035, "V", "Brief File 4: 2026-2035")),
    ("S0 at the current pace", assume("Year 100% at unadjusted S0 pace", must_do_year, "D", f"{remaining} non-zero-emission buses / {ze_per_year:.0f} per year")),
], columns=["anchor", "year"])
display(TIMELINE)

check("Fleet split adds up", (pace_tbl.zero_emission + pace_tbl.not_zero_emission == pace_tbl.fleet_total).all())
check("Zero-emission fleet grew 2020-25", ze_per_year > 0, f"{ze_per_year:.1f} buses/year")

,fleet_total,zero_emission,not_zero_emission
2020-12-31,2066,76,1990
2025-12-31,2210,462,1748


,anchor,year
0,S1 target,2030
1,Madrid official target,2033
2,File 4 horizon,2035
3,S0 at the current pace,2048


CHECK PASS | Fleet split adds up
CHECK PASS | Zero-emission fleet grew 2020-25 | 77.2 buses/year


np.True_

## Q1 · How does the current fleet work? [ONCE]
### 1a · Buses
Fleet in service on 31 Dec 2025 by fuel and size code, with age. Size codes decoded as in main notebook §4: `02` is the only code whose models are articulated (`…-ART`, 118–122 places); `01` is the standard 12 m bus; `03`, `04` and `06` are midi and mini buses, counted as Standard because the brief has only two types.

In [18]:
fleet25 = snap["2025-12-31"]
fleet25 = fleet25.assign(size=np.where(fleet25.size_code == "02", "Articulated", "Standard"))   # main notebook §4
display(pd.crosstab(fleet25.fuel, fleet25.size_code, margins=True, margins_name="total"))
display(pd.crosstab(fleet25.fuel, fleet25["size"], margins=True, margins_name="total"))
display(fleet25.groupby("fuel").agg(buses=("vehicle_id", "count"), median_capacity=("capacity", "median"),
                                    mean_age=("age_years", "mean"), oldest=("age_years", "max")).round(1))

size_code,01,02,03,04,06,total
fuel,,,,,,
Battery-electric,408,0,20,0,24,452
CNG,1650,85,0,13,0,1748
Hydrogen fuel cell,10,0,0,0,0,10
total,2068,85,20,13,24,2210


size,Articulated,Standard,total
fuel,,,
Battery-electric,0,452,452
CNG,85,1663,1748
Hydrogen fuel cell,0,10,10
total,85,2125,2210


,buses,median_capacity,mean_age,oldest
fuel,,,,
Battery-electric,452,74.0,2.9,7.9
CNG,1748,78.0,6.6,15.1
Hydrogen fuel cell,10,74.0,0.9,0.9


**Age profile.** The CNG fleet's age drives retirements in File 4: older buses are replaced first.

In [19]:
px.histogram(fleet25, x="age_years", color="fuel", nbins=20, barmode="stack",
             title="Fleet in service on 31 Dec 2025: age by fuel", labels={"age_years": "age (years)"}).show()

### 1b · Lines
Top lines by working-day riders, from the feature table (section 4). Electric / gas status per line: **dropped** here (the open data has no bus-to-line link); notebook 01 §7 uses the six lines with line-level evidence. Km and schedule per line: notebook 01 §4 (File 2).

In [20]:
display(features.sort_values("riders_per_day", ascending=False)
        [["peak_buses", "riders_per_day", "riders_per_bus_hour", "span_hours", "peak_to_average"]].head(15))
print(f"{len(features)} lines with supply records; {len(only_riders)} more have riders only: {only_riders}")

,peak_buses,riders_per_day,riders_per_bus_hour,span_hours,peak_to_average
line_id,,,,,
034,21.30,32876.04,100.03,20.00,1.30
881,13.36,28468.28,130.44,21.07,1.29
070,18.68,26815.64,104.24,20.00,1.45
027,21.61,25655.62,92.16,19.99,1.55
069,20.57,24421.44,78.85,20.00,1.33
068,21.19,23997.24,74.06,19.01,1.24
883,19.15,22271.10,61.88,21.20,1.13
028,18.84,21082.64,82.21,19.99,1.47
021,17.95,19667.44,73.98,19.99,1.35


240 lines with supply records; 8 more have riders only: ['741', '809', '841', '886', '887', '888', '889', '890']


### 1c · Depots
Grid capacity and electrical output per depot. Answered in notebook 01 §5–§6 (File 3): Sanchinarro (8 MW) and La Elipa (20 MW) are evidenced, both planned; the other three are estimated from charging points.

In [21]:
F = {n: pd.read_csv(os.path.join(OUT, n), encoding="utf-8", dtype={"line_id": str}) for n in ("File_1.csv", "File_2.csv", "File_3.csv", "File_4.csv")}
display(F["File_3.csv"][["depot_name", "buses_assigned", "peak_mw_managed", "connection_mw_estimate", "connection_evidence", "readiness_status"]].round(1))


,depot_name,buses_assigned,peak_mw_managed,connection_mw_estimate,connection_evidence,readiness_status
0,Carabanchel,447,14.6,19.8,Estimated,Ready
1,Fuencarral,518,20.1,4.3,Estimated,Critical
2,La Elipa,290,10.4,20.0,Evidenced,Ready
3,Entrevías,426,13.5,0.9,Estimated,Critical
4,Sanchinarro,353,13.0,8.0,Evidenced,Critical


## Q2 · Apply the Madrid setup to each benchmark city [PER CITY]
Answered by File 1 (notebook 01 §9): Helsinki, Paris, London, Copenhagen and Berlin; Brussels is not covered. The lines-electrified KPI is **dropped**: no peer publishes zero-emission lines on a comparable basis, so the fleet share is used.

**KPI definition from the board:** % electrified = # zero-emission lines ÷ total # lines (alongside the fleet share used in File 1).

In [22]:
b = F["File_1.csv"][["city", "fleet_total", "ze_fleet", "ze_share_pct", "ze_target_year", "charging_model"]].copy()
b["gap_to_madrid_pts"] = (b.ze_share_pct - float(b.loc[b.city == "Madrid", "ze_share_pct"].iloc[0])).round(1)
display(b)


,city,fleet_total,ze_fleet,ze_share_pct,ze_target_year,charging_model,gap_to_madrid_pts
0,Helsinki,1154,600,52.0,2035.0,Mixed,31.1
1,Paris,4700,1100,23.4,NaN,Depot,2.5
2,London,8797,1972,22.4,2034.0,Depot,1.5
3,Copenhagen,274,219,79.9,2025.0,Depot,59.0
4,Berlin,1532,277,18.1,NaN,Mixed,-2.8
5,Madrid,2210,462,20.9,2033.0,Depot,0.0


## Q3 · Why were certain lines electrified first? [ONCE]
**Dropped**: only six lines have line-level evidence of being fully electric, too few to compare against the rest, and traffic and incline were not sourced. Notebook 01 §7 ranks lines on CO₂, riders per peak bus, feasibility and depot readiness instead.

In [23]:
print("Q3 dropped: six evidenced electric lines (001, 002, 22, 87, 108, 121) are too few for a comparison; see notebook 01 section 7")


Q3 dropped: six evidenced electric lines (001, 002, 22, 87, 108, 121) are too few for a comparison; see notebook 01 section 7


**External signals.** Dropped: traffic and incline were not sourced.

In [24]:
print("External signals (traffic, incline): dropped, not sourced")


External signals (traffic, incline): dropped, not sourced


## Q4 · How would we transition the buses? [ONCE]
**a) Grid electrification / depots:** depot information, gap to electrification per depot, route impact / coverage.
**b) Which routes are replaced:** phases, route impact (buses, bus metrics).

Answered in notebook 01 §6 (needed MW per depot) and §7 (File 2 years). Gap per depot below.

In [25]:
d = F["File_3.csv"].assign(gap_mw=lambda x: (x.peak_mw_managed - x.connection_mw_estimate).clip(lower=0).round(1))
display(d[["depot_name", "n_lines_assigned", "buses_assigned", "peak_mw_managed", "connection_mw_estimate", "gap_mw", "readiness_status", "conversion_year"]].round(1))


,depot_name,n_lines_assigned,buses_assigned,peak_mw_managed,connection_mw_estimate,gap_mw,readiness_status,conversion_year
0,Carabanchel,58,447,14.6,19.8,0.0,Ready,2029
1,Fuencarral,56,518,20.1,4.3,15.8,Critical,2030
2,La Elipa,31,290,10.4,20.0,0.0,Ready,2030
3,Entrevías,47,426,13.5,0.9,12.6,Critical,2029
4,Sanchinarro,42,353,13.0,8.0,5.0,Critical,2029


**Routes replaced, by phase.** Lines grouped by File 2's electrification year (S1), with the buses and km converted in each phase.

In [26]:
display(F["File_2.csv"].groupby("electrification_year").agg(lines=("line_id", "count"), peak_buses=("peak_buses", "sum"),
                                                            daily_km=("daily_km", "sum")).round(0).T)


electrification_year,2025,2026,2027,2028,2029,2030
lines,6.0,5.0,47.0,28.0,61.0,87.0
peak_buses,47.0,79.0,366.0,367.0,367.0,808.0
daily_km,6318.0,12401.0,55034.0,60017.0,61599.0,146062.0


## Q5 · What does implementing electrification mean? [ONCE]
Finance / costs: bus purchases (`P['bus_price_eur']`), age and retirement of the gas fleet, gas cost, labour.

Answered in notebook 01 §8 (File 4 capex, retirements, gas exposure) and notebook 04 §10 (financing). Labour cost is **dropped** (no source).

In [27]:
display(F["File_4.csv"].pivot(index="year", columns="scenario", values=["capex_meur", "n_cng", "cng_t"]).round(1))


capex_meur          n_cng            cng_t         
scenario          0      1       0       1        0        1
year                                                        
2026           61.7   61.7  1678.0  1678.0  44189.0  44189.0
2027           89.4  277.0  1613.0  1357.0  42447.8  39145.9
2028           92.2  274.1  1536.0  1034.0  40616.3  30839.5
2029           87.7  236.6  1459.0   711.0  38629.9  22507.3
2030           67.6  515.5  1349.0     0.0  36218.0   9170.6
2031          263.3   10.5   984.0     0.0  30091.4      0.0
2032          179.2   11.7   753.0     0.0  22404.1      0.0
2033          188.6   19.7   520.0     0.0  16419.3      0.0
2034          105.5   34.4   422.0     0.0  12150.1      0.0
2035          255.5   30.8   112.0     0.0   6887.6      0.0

## Q6 · Scenario comparison [ONCE]
Compare the timeline anchors (2030 ideal · 2033 wildcard · 2035 mid · must-do year) on: depot → line, area / zones, bus ages / lifetime, distance, emissions, ridership.

Answered in notebook 01 §8: File 4 (S0, S1) and the earliest-credible-year run (2030 not achievable, 2033 earliest credible, 2037–2040 expected). Zones and ridership per anchor are **dropped**.

In [28]:
display(F["File_4.csv"].pivot(index="year", columns="scenario", values=["ze_share_pct", "co2_t", "co2_avoided_t"]).round(1))


ze_share_pct            co2_t           co2_avoided_t          
scenario            0      1         0         1             0         1
year                                                                    
2026             24.1   24.1  124451.5  124451.5        2139.6    2139.6
2027             27.0   38.6  120325.1  112500.1        6266.1   14091.1
2028             30.5   53.2  115984.7   92815.4       10606.5   33775.8
2029             34.0   67.8  111277.4   73069.6       15313.7   53521.6
2030             39.0  100.0  105561.5   41464.0       21029.6   85127.2
2031             55.5  100.0   91042.6   19731.4       35548.6  106859.8
2032             65.9  100.0   72825.0   19731.4       53766.1  106859.8
2033             76.5  100.0   58642.3   19731.4       67948.9  106859.8
2034             80.9  100.0   48524.8   19731.4       78066.4  106859.8
2035             94.9  100.0   36053.8   19731.4       90537.4  106859.8

## Visualisation
Board list: scenario planning (2030 / 2035 / wildcard), map and routes with lines, bus scheduling (peak / off-peak, days of the week, times), future state over time (costs, emissions, line transition). The bus-scheduling view is below; the other views are in notebook 01 §10 (HTML and static PNGs in `outputs/`).

In [29]:
by_hour = (supply.groupby(["day_code", "date", "hour_band"]).buses_in_service.sum()
           .groupby(["day_code", "hour_band"]).mean().rename("buses_in_service").reset_index())
by_hour["day_type"] = by_hour.day_code.map(DAY)
px.line(by_hour, x="hour_band", y="buses_in_service", color="day_type", markers=True,
        title="Average buses in service by hour, Jan-Aug 2026", labels={"hour_band": "hour band (24+ = after midnight)"}).show()
peak_hour = by_hour.loc[by_hour[by_hour.day_code == "LA"].buses_in_service.idxmax()]
print(f"Working-day peak: {peak_hour.buses_in_service:.0f} buses in hour band {peak_hour.hour_band}")

Working-day peak: 1692 buses in hour band 9


**Remaining views.** Drawn in notebook 01 §10: route map by charging strategy, depot readiness, benchmark and S0 vs S1 (PNGs in `outputs/`).

In [30]:
print("Static views in outputs/:", sorted(f for f in os.listdir(OUT) if f.endswith(".png")))


Static views in outputs/: ['view_i_routes_by_charging_strategy.png', 'view_ii_depot_readiness.png', 'view_iii_benchmark.png', 'view_iv_scenarios.png']


## Checks, assumptions and sources

In [31]:
display(pd.DataFrame(CHECKS)); display(pd.DataFrame(ASSUMPTIONS)); show_register()

,check,result,detail
0,Fleet: one record per vehicle,PASS,"5351 records, 5351 vehicles"
1,supply: no duplicate keys,PASS,
2,riders: no duplicate keys,PASS,
3,calendar: no duplicate keys,PASS,
4,Every fleet fuel code has a label,PASS,
5,Every calendar day has a label,PASS,
6,Every ridership date is in the calendar,PASS,
7,Hour bands share one scale (0-31),PASS,
8,Line features: every supply line has riders,PASS,240 of 240 lines
9,Fleet split adds up,PASS,


,variable,value,label,evidence,impact,sensitivity
0,CO2 factor CNG (kg CO2 per kg),2.7175,V,MITECO ETS2 calculation factors: natural gas 5...,"Files 2, 4",56.06 t/TJ (road-transport value): -0.1 %
1,CO2 factor electricity Spain 2025 (g CO2 per kWh),108.4,V,"Red Electrica REData, national 2025, 'tCO2 eq....","Files 1, 2, 4",Declining grid path: notebook 04
2,Fleet near-duplicate rule,"one record per vehicle, highest capacity kept;...",A,Near-duplicates differ mainly in capacity and ...,,
3,Fuel code labels,"{'GN': 'CNG', 'EL': 'Battery-electric', 'H': '...",P,"EL, GN and H match EMT's published fleet mix; ...",,
4,Zero-emission fuel codes,"(EL, H)",A,Battery-electric and hydrogen only; CNG and bi...,,
5,Day-type labels,"{'LA': 'Working day', 'SA': 'Saturday', 'FE': ...",P,"LA, SA, FE documented in the supply README; th...",,
6,"Zero-emission buses added per year, 2021-25 (u...",77.2,D,"Fleet history, active vehicles on 31 Dec 2020 ...",Timeline,Main notebook §3 reconciliation
7,S1 target year,2030,V,Brief scenario S1,,
8,Madrid official target year,2033,A,"City of Madrid target, Aug 2023; to verify it ...",,
9,File 4 horizon,2035,V,Brief File 4: 2026-2035,,


,file,url,downloaded,bytes,sha256_12,obtained,note
0,emt_gtfs.zip,https://datos.emtmadrid.es/dataset/9b23259a-44...,2026-11-26,21044007,62453fb95492,script,"EMT GTFS, rolling daily feed — https://datos.m..."
1,emt_supply_2026.csv,https://datos.emtmadrid.es/dataset/03c64a21-3e...,2026-11-26,21555684,9fdcee98ef74,script,"Buses in service per day, line and hour band, ..."
2,emt_fleet_history.csv,https://datos.emtmadrid.es/dataset/071db69e-5e...,2026-11-26,1109695,ff90c42290bd,script,"Fleet history (vehicle, dates, fuel, size) — h..."
3,emt_calendar.csv,https://datos.emtmadrid.es/dataset/405a6ed0-94...,2026-11-26,56877,b967eda5e8d5,script,Operating calendar 2024-2027 — https://datos.m...
4,emt_ridership_2026.csv,https://datos.emtmadrid.es/dataset/29b27217-42...,2026-11-26,24395594,d3bffd83ce3e,script,"Optional: daily riders per line and hour band,..."
